In [1]:
import duckdb, os

src = "../../data/AZ_harmonized_data"
dst = "../../data/parquet"

for name in ["fact_expression_depmap", "fact_expression_hpa", "geo",
             "fact_proteomics", "fact_mutations", "fact_fusions", "dim_cell_lines"]:
    print(name, end=" ... ")
    duckdb.sql(f"""
        COPY (SELECT * FROM read_csv_auto('{src}/{name}.csv'))
        TO '{dst}/{name}.parquet' (FORMAT PARQUET)
    """)
    print("done")

fact_expression_depmap ... 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

done
fact_expression_hpa ... 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

done
geo ... 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

done
fact_proteomics ... done
fact_mutations ... done
fact_fusions ... done
dim_cell_lines ... done


In [6]:
import pandas as pd

src = "../../processed"
dst = "../../processed"

# one row per cell line already, only the id column needs renaming
sig = pd.read_parquet(f"{src}/signatures_clean.parquet").reset_index()
sig = sig.rename(columns={"ModelID": "ach_id"})
sig.to_csv(f"{dst}/fact_signatures.csv", index=False)

# metabolomics: wide (225 metabolite columns) -> long
metab = pd.read_parquet(f"{src}/metabolomics_clean.parquet")
metab = metab.drop(columns=["CCLE_ID"]).rename(columns={"DepMap_ID": "ach_id"})
metab_long = metab.melt(id_vars="ach_id", var_name="metabolite", value_name="value")
metab_long.to_csv(f"{dst}/fact_metabolomics.csv", index=False)

# mirna: miRNAs as rows, cell lines as columns -> transpose then long
mirna = pd.read_parquet(f"{src}/mirna_clean.parquet").T
mirna.index.name = "ach_id"
mirna_long = mirna.reset_index().melt(id_vars="ach_id", var_name="mirna_id", value_name="value")
mirna_long.to_csv(f"{dst}/fact_mirna.csv", index=False)

for f in ["fact_signatures", "fact_metabolomics", "fact_mirna"]:
    df = pd.read_csv(f"{dst}/{f}.csv")
    print(f, df.shape, df.columns.tolist())

fact_signatures (1955, 7) ['ach_id', 'MSIScore', 'LoHFraction', 'WGD', 'CIN', 'Ploidy', 'Aneuploidy']
fact_metabolomics (208800, 3) ['ach_id', 'metabolite', 'value']
fact_mirna (698768, 3) ['ach_id', 'mirna_id', 'value']
